# Nuer Whisper ASR Fine-tuning

Fine-tune Whisper on Nuer language data using Google Colab's free GPU.

**Setup:**
1. Upload your `nuer_whisper_asr` folder to Google Drive
2. Run all cells in order
3. Model will be saved to your Drive

## Step 1: Mount Google Drive
Connect your Drive where the data is stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate torchaudio sentencepiece protobuf
!pip install -q evaluate jiwer  # For WER metric

print("✓ Dependencies installed")

## Step 3: Set Paths

**IMPORTANT:** Update `DRIVE_PATH` to where you uploaded your project.

In [ ]:
import os
from pathlib import Path

# UPDATE THIS to your project location in Drive
# Example: /content/drive/MyDrive/nuer_whisper_asr
DRIVE_PATH = Path("/content/drive/MyDrive/nuer_whisper_asr")

# Check if path exists
if not DRIVE_PATH.exists():
    print(f"❌ Path not found: {DRIVE_PATH}")
    print("Available files in Drive:")
    !ls -la /content/drive/MyDrive/ | head -20
else:
    print(f"✓ Project found at: {DRIVE_PATH}")
    print(f"  Data: {DRIVE_PATH / 'data'}")
    print(f"  Audio: {DRIVE_PATH / 'voice dataset'}")

## Step 4: Load and Verify Data

In [ ]:
import json

# Load data
DATA_DIR = DRIVE_PATH / "data"
TRAIN_FILE = DATA_DIR / "train.json"
TEST_FILE = DATA_DIR / "test.json"

with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"✓ Train samples: {len(train_data)}")
print(f"✓ Test samples: {len(test_data)}")

# Verify audio paths
AUDIO_DIR = DRIVE_PATH / "voice dataset"
sample_audio = AUDIO_DIR / train_data[0]['audio'].replace('../voice dataset/', '')

if sample_audio.exists():
    print(f"✓ Audio files accessible: {sample_audio.name}")
else:
    print(f"❌ Audio not found: {sample_audio}")

## Step 5: Prepare Dataset for HuggingFace

In [ ]:
from datasets import Dataset, Audio

def prepare_dataset(data, base_dir=AUDIO_DIR):
    """Prepare dataset for HuggingFace."""
    audio_paths = []
    texts = []
    
    for item in data:
        # Convert relative path to absolute
        rel_path = item["audio"].replace('../voice dataset/', '')
        audio_path = str(base_dir / rel_path)
        
        if os.path.exists(audio_path):
            audio_paths.append(audio_path)
            texts.append(item["text"])
    
    dataset = Dataset.from_dict({
        "audio": audio_paths,
        "text": texts
    })
    
    # Cast audio column to Audio type (resamples to 16kHz)
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    
    return dataset

print("Preparing datasets...")
train_dataset = prepare_dataset(train_data)
test_dataset = prepare_dataset(test_data)

print(f"✓ Train: {len(train_dataset)} samples")
print(f"✓ Test: {len(test_dataset)} samples")

## Step 6: Load Whisper Model

In [ ]:
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration
)

MODEL_NAME = "openai/whisper-small"  # Options: tiny, base, small, medium

print(f"Loading {MODEL_NAME}...")

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_NAME, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Disable forced decoder IDs for fine-tuning
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

print(f"✓ Model loaded: {model.num_parameters()/1e6:.1f}M parameters")

## Step 7: Preprocess Dataset

In [ ]:
def prepare_dataset_batch(batch):
    """Process a batch of data."""
    # Load audio
    audio = batch["audio"]
    
    # Compute log-Mel input features
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    
    # Encode target text
    batch["labels"] = tokenizer(batch["text"]).input_ids
    
    return batch

print("Processing train dataset...")
train_dataset = train_dataset.map(prepare_dataset_batch, remove_columns=train_dataset.column_names)

print("Processing test dataset...")
test_dataset = test_dataset.map(prepare_dataset_batch, remove_columns=test_dataset.column_names)

print("✓ Preprocessing complete")

## Step 8: Setup Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("✓ Data collator ready")

## Step 9: Define WER Metric

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # Replace -100 with pad token id
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("✓ WER metric ready")

## Step 10: Configure Training

In [ ]:
from transformers import Seq2SeqTrainingArguments

# Create output directory in Drive
OUTPUT_DIR = DRIVE_PATH / "models" / "whisper-nuer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
    save_total_limit=2,  # Keep only 2 checkpoints to save space
)

print(f"✓ Training config ready")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Steps: {training_args.max_steps}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")

## Step 11: Start Training 🚀

In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n" + "="*50)
print("Starting training...")
print("="*50 + "\n")

trainer.train()

print("\n" + "="*50)
print("Training complete!")
print("="*50)

## Step 12: Save Final Model

In [ ]:
FINAL_MODEL_DIR = DRIVE_PATH / "models" / "whisper-nuer-final"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Saving model...")
trainer.save_model(str(FINAL_MODEL_DIR))
processor.save_pretrained(str(FINAL_MODEL_DIR))

print(f"✓ Model saved to: {FINAL_MODEL_DIR}")
print(f"\nModel size: {sum(f.stat().st_size for f in FINAL_MODEL_DIR.rglob('*') if f.is_file()) / 1e6:.1f} MB")

## Step 13: Test the Model

In [ ]:
import torchaudio

# Load a test sample
test_sample = test_data[0]
test_audio_path = AUDIO_DIR / test_sample['audio'].replace('../voice dataset/', '')

print(f"Testing on: {test_sample['text']}")
print(f"Audio: {test_audio_path.name}")

# Load audio
waveform, sample_rate = torchaudio.load(str(test_audio_path))

# Process
input_features = processor(
    waveform.squeeze().numpy(),
    sampling_rate=sample_rate,
    return_tensors="pt"
).input_features

# Generate
predicted_ids = model.generate(input_features.to(model.device))
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(f"\nGround truth: {test_sample['text']}")
print(f"Prediction:   {transcription}")

## Next Steps

1. **Download model** (if needed locally):
   - Find it in your Google Drive: `models/whisper-nuer-final/`

2. **Use for inference**:
   - Load with: `pipeline('automatic-speech-recognition', model=path)`

3. **Quantize for mobile** (future step):
   - Convert to ONNX or TFLite for edge deployment